# **Deep Natural Language Processing @ PoliTO**

---


**Teaching Assistant:** Giuseppe Gallipoli

**Credits:** Moreno La Quatra

**Practice 6:** Automatic Text Summarization - Extractive (part 1) and Abstractive (part 2) Summarization

# Automatic Text Summarization

Automatic text summarization is the task of producing a concise and fluent summary while preserving key information content and overall meaning. The summarization task is challenging because it requires a deep understanding of the text, including both its content and its style.

**Extractive Summarization** is the task of selecting a subset of the original text to form the summary. The selected sentences are concatenated to form the summary. Extractive summarization is the most common approach to text summarization as it only requires to understand the content of the text and estimate the importance of each sentence. It does not require to **generate** new text, which is a more complex task and requires both a deeper understanding of the text and text generation capabilities.

**Abstractive Summarization** is the task of generating a summary that is not a direct copy of the original text. It requires a deeper understanding of the text and the ability to generate new text. The model needs to understand the content of the text and the style of the author to produce a summary that is fluent and coherent with the original text.

## Abstractive Text Summarization

Abstractive summarization models build an internal semantic representation of the original content, and then use this representation to generate a summary. The main difference with extractive summarization is that abstractive summarization models do not select sentences from the original text, but they generate new sentences.
Abstraction may transform the extracted content by paraphrasing sections of the source document, to condense a text more strongly than extraction. Such transformation, however, is computationally much more challenging than extraction and requires a more sophisticated model.

![https://dbdmg.polito.it/dbdmg_web/wp-content/uploads/2025/11/ext_abs_summ.png](https://dbdmg.polito.it/dbdmg_web/wp-content/uploads/2025/11/ext_abs_summ.png)

Similarly to the previous section of the practice, we use the BBC News Summary dataset available in [Kaggle](https://www.kaggle.com/pariza/bbc-news-summary). You don't need to download the dataset again, you can use the one you already downloaded in the previous part.

In [1]:
%%capture
!wget https://github.com/MorenoLaQuatra/DeepNLP/raw/main/practices/P5/bbc_news.zip
!unzip bbc_news.zip

### **Question 1: Split data collection**

The data collection contains news articles belonging to different categories (e.g., business, sport, tech, etc.) and the corresponding summaries. In this question you will split the data collection into training, validation and test sets. The training set will be used to train the model, the validation set will be used to select the best model and the test set will be used to evaluate the final model. The goal is to stratify the data collection by category, so that each category is represented in the same proportion in each set. Be sure to select 10% of the data **of each category** for the test set. The remaining data can be split according to your preference.

**Note 1:** Some files can report UnicodeError, feel free to ignore it (`errors` parameter).
```python
f = open(FILENAME, 'r', encoding='utf-8', errors='ignore')
```

**Note 2:** You can fix encoding after file reading by using [ftfy](https://pypi.org/project/ftfy/) library.

```python
import ftfy
fixed_text = ftfy.fix_text(text)
```

The following cell installs the ftfy library that can be used to fix encoding issues.

In [2]:
%%capture
!pip install ftfy rouge rouge_score

In [3]:
# your code here
import os
import pandas as pd
from sklearn.model_selection import train_test_split

base_path = "/kaggle/input/bbc-news-summary/BBC News Summary/News Articles/"
summary_path = "/kaggle/input/bbc-news-summary/BBC News Summary/Summaries"
data = {'text': [], 'summary': [], 'category': []}

if os.path.exists(base_path):
    for cat in os.listdir(base_path):
        cat_path = os.path.join(base_path, cat)
        sum_cat_path = os.path.join(summary_path, cat)

        if os.path.isdir(cat_path):
            for file in os.listdir(cat_path):
                file_path = os.path.join(cat_path, file)
                sum_file_path = os.path.join(sum_cat_path, file)

                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()
                    data['text'].append(text)
                    data['category'].append(cat)
                    with open(sum_file_path, 'r', encoding='utf-8', errors='ignore') as s:
                        summary = s.read()
                        data['summary'].append(summary)

    df = pd.DataFrame(data)
    print(df.head())
else:
    print(f"The folder does not exists.")

train_val_df, test_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df['category']
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=1/9,
    random_state=42,
    stratify=train_val_df['category']
)

                                                text  \
0  Budget to set scene for election\n\nGordon Bro...   
1  Army chiefs in regiments decision\n\nMilitary ...   
2  Howard denies split over ID cards\n\nMichael H...   
3  Observers to monitor UK election\n\nMinisters ...   
4  Kilroy names election seat target\n\nEx-chat s...   

                                             summary  category  
0  - Increase in the stamp duty threshold from £6...  politics  
1  "They are very much not for the good and will ...  politics  
2  Michael Howard has denied his shadow cabinet w...  politics  
3  The report said individual registration should...  politics  
4  UKIP's leader, Roger Knapman, has said he is g...  politics  


### **Question 2: BART (pretrained) seq2seq model**

[BART](https://arxiv.org/abs/1910.13461) is a sequence-to-sequence model trained with denoising as pretraining objective. BART is based on the [Transformer](https://arxiv.org/abs/1706.03762) architecture and it is trained on a large amount of text data.

The [HuggingFace transformers](https://huggingface.co/transformers/) library provides pretrained BART models that can be used to generate summaries. For this question you will use the [BART model pre-trained on CNN-DailyMail dataset](https://huggingface.co/facebook/bart-large-cnn) to summarize the articles in the BBC test set.
You can use the `pipeline` function to create a summarization pipeline. The pipeline takes as input the text to summarize and returns the summary.

**Note 1**: for generating summaries, set the maximum length to 100 and the minimum length to your preferred value (if you set the minimum length to a very low value, the model may generate summaries that are too short).

**Note 2**: to **speed up computation**, you can use the distilled version of the BART model (e.g., `sshleifer/distilbart-cnn-12-6` available [here](https://huggingface.co/sshleifer/distilbart-cnn-12-6)). Please note that the distilled version can be less effective than the larger version.

**Note 3**: You can use the [summarization pipeline](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.SummarizationPipeline). Explictly set truncation to True to avoid index errors (e.g., `summarizer(..., truncation=True)`).

**Note 4**: If you have a GPU runtime on Colab, you can use it to speed up computation. To use the GPU with the pipeline, you can set the `device` parameter to `0` (e.g., `pipeline(..., device=0)`).


In [4]:
%%capture
!pip install transformers=4.57.1
!pip install bitsandbytes

In [6]:
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from transformers.pipelines.pt_utils import KeyDataset
from tqdm.auto import tqdm
from datasets import Dataset

test_dataset = Dataset.from_pandas(test_df)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_id = "facebook/bart-large-cnn"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    truncation=True,
    model_max_length=1024
)

pipe = pipeline(
    "summarization",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    batch_size=8
)

generated_summaries = []

for out in tqdm(pipe(KeyDataset(test_dataset, "text"), 
                     truncation=True, 
                     max_length=100, 
                     min_length=30, 
                     batch_size=8), total=len(test_dataset)):
    
    generated_summaries.append(out[0]['summary_text'])

test_dataset = test_dataset.add_column("generated_summary", generated_summaries)

df_view = pd.DataFrame(test_dataset[0:5])

print(df_view[['summary', 'generated_summary']])

Device set to use cuda:0


  0%|          | 0/223 [00:00<?, ?it/s]

                                             summary  \
0  Andy Roddick was the only player to put up any...   
1  Mortgage lending rose by £7.1bn in December, u...   
2  Mr Bannatyne has previously given Labour £50,0...   
3  Mark Lewis-Francis says his Olympic success ha...   
4  An attachment in the e-mail contains the virus...   

                                   generated_summary  
0  The last year has seen one player dominate men...  
1  New loans in December rose to 83,000, slightly...  
2  Donor attacks Blair-Brown 'feud' and governmen...  
3  The Birchfield Harrier pipped Maurice Greene o...  
4  The e-mails show that they have come from an f...  


In [5]:
%%capture
!pip install evaluate

In [7]:
import evaluate

rouge = evaluate.load('rouge')

results = rouge.compute(
    predictions=test_dataset['generated_summary'],
    references=test_dataset['summary']
)

print(results)

ValueError: pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility. Expected 112 from C header, got 104 from PyObject

### **Question 3: Try with another pretrained seq2seq model**

Now try using a different (pretrained) sequence-to-sequence model. You can choose any model you prefer, for example you may look at the most downloaded models for text summarization on [Hugging Face](https://huggingface.co/models?pipeline_tag=summarization&sort=downloads), e.g., `facebook/bart-large-cnn`.

Evaluate the performance of the new model you selected and compare it with the results obtained in the previous exercise (Q2).

How does performance vary depending on the pretrained model used? Which factors influence it the most?\
If you wish, you can also repeat the exercise trying a different model architecture (e.g., T5).

In [7]:
# your code here
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from transformers.pipelines.pt_utils import KeyDataset
from tqdm.auto import tqdm
from datasets import Dataset

test_dataset = Dataset.from_pandas(test_df)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_id = "google-t5/t5-small"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id
)

pipe = pipeline(
    "summarization",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    batch_size=8
)

generated_summaries = []

for out in tqdm(pipe(KeyDataset(test_dataset, "text"), 
                     truncation=True, 
                     max_new_tokens=100, 
                     min_length=30, 
                     batch_size=8), total=len(test_dataset)):
    
    generated_summaries.append(out[0]['summary_text'])

test_dataset = test_dataset.add_column("generated_summary", generated_summaries)

df_view = pd.DataFrame(test_dataset[0:5])

print(df_view[['summary', 'generated_summary']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


  0%|          | 0/223 [00:00<?, ?it/s]

                                             summary  \
0  Andy Roddick was the only player to put up any...   
1  Mortgage lending rose by £7.1bn in December, u...   
2  Mr Bannatyne has previously given Labour £50,0...   
3  Mark Lewis-Francis says his Olympic success ha...   
4  An attachment in the e-mail contains the virus...   

                                   generated_summary  
0  Roger Federer became the first man since Mats ...  
1  mortgage lending rose by £7.1bn in December, u...  
2  a labour donor says he will almost certainly r...  
3  mark Lewis-Francis has yet to decide what even...  
4  the e-mails show that they have come from an f...  


In [8]:
import evaluate

rouge = evaluate.load('rouge')

results = rouge.compute(
    predictions=test_dataset['generated_summary'],
    references=test_dataset['summary']
)

print(results)

{'rouge1': 0.30423107932723387, 'rouge2': 0.2025338549434676, 'rougeL': 0.22866963809803859, 'rougeLsum': 0.22898970513732775}


### **Question 4: Finetuning seq2seq model**

The BBC dataset is provided with a training set. You can use the training set to finetune a BART model to generate summaries. You can use the [transformers library](https://huggingface.co/transformers/) to finetune the model.
You have an example of how to finetune a model for sequence classification in the Transformers Overview notebook (see EA1): you can use it as a reference to adapt it to the summarization case.

You can use the [`datasets` library](https://huggingface.co/docs/datasets/) and [Trainer API](https://huggingface.co/transformers/training.html#fine-tuning-in-pytorch-with-the-trainer-api) for finetuning and [`evaluate` library](https://huggingface.co/docs/evaluate/index) to evaluate the model.

Alternatively, even in this case, you can evaluate the model using ROUGE-2 precision, recall and F1-score. You may want to use the [`compute_metrics`](https://huggingface.co/course/chapter3/3?fw=pt#evaluation) function to monitor the ROUGE-2 scores during training and select the best model according to the ROUGE-2 scores on the validation set.

In [ ]:
!pip install datasets evaluate transformers

In [ ]:
# your code here
import torch
import numpy as np
import evaluate
from tqdm.auto import tqdm
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)

model_id = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, device_map="auto")

def prepare_dataset(df, tokenizer, max_input=512, max_target=100):
    input_ids = []
    attention_masks = []
    labels = []
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        inputs = tokenizer(
            row['text'], 
            max_length=max_input, 
            truncation=True,
            padding=False 
        )
        
        targets = tokenizer(
            text_target=row['summary'], 
            max_length=max_target, 
            truncation=True,
            padding=False
        )
        
        input_ids.append(inputs['input_ids'])
        attention_masks.append(inputs['attention_mask'])
        labels.append(targets['input_ids'])
        
    return Dataset.from_dict({
        'input_ids': input_ids,
        'attention_mask': attention_masks,
        'labels': labels
    })


train_dataset = prepare_dataset(train_df, tokenizer)
val_dataset = prepare_dataset(val_df, tokenizer)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    return {k: round(v, 4) for k, v in result.items()}

args = Seq2SeqTrainingArguments(
    output_dir="bart-finetuned-bbc",
    learning_rate=2e-5,           
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=1,           
    num_train_epochs=1,           
    predict_with_generate=True,
    fp16=True,                    
    logging_steps=50,
    report_to="none"              
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from transformers import pipeline
from tqdm.auto import tqdm
import pandas as pd

finetuned_pipe = pipeline(
    "summarization",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    batch_size=8
)

finetuned_summaries = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    out = finetuned_pipe(
        row['text'], 
        truncation=True, 
        max_length=100, 
        min_length=30
    )
    finetuned_summaries.append(out[0]['summary_text'])

rouge = evaluate.load('rouge')
results_finetuned = rouge.compute(
    predictions=finetuned_summaries,
    references=test_df['summary'].tolist()
)

print(results_finetuned)

### **Bonus**: Upload **your** model to the [HuggingFace model hub](https://huggingface.co/models) and share it

The HuggingFace model hub is a repository of pretrained models that can be used to perform a wide range of NLP tasks. You can upload your model to the hub and share it with the community. You can find more information about the model hub [here](https://huggingface.co/docs/hub/main).

**Note 1**: If you want to extend the practice, you can try to finetune a BART model on other data collections that are available online or on the [HuggingFace datasets hub](https://huggingface.co/datasets) and share the results and the model.

**How to upload a model to the HuggingFace model hub**

- **Step 1**: Create a [HuggingFace account](https://huggingface.co/join)
- **Step 2**: Login to your account from the notebook using the token provided in the account page ([more info](https://huggingface.co/settings/tokens))
```python
from huggingface_hub import notebook_login
notebook_login()
```
- **Step 3**: Prepare the model and tokenizer for the upload
```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained("YOUR_LOCAL_FOLDER/checkpoint-...") # load the model
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
```
- **Step 4**: Install git-lfs on Colab
```python
!curl -s https://packagecloud.io/install/repositories/github/git-lfs/script.deb.sh | sudo bash
!sudo apt-get install git-lfs
!git lfs install
!git config --global credential.helper store
```

- **Step 5**: Upload the model to the HuggingFace model hub
```python
MODEL_NAME = `my-awesome-model-name`
tokenizer.push_to_hub(MODEL_NAME, use_temp_dir=True)
finetuned_model.push_to_hub(MODEL_NAME, use_temp_dir=True)
```

The model will be uploaded to the [HuggingFace model hub](https://huggingface.co/models) and you can share the link to the model with the community. If you want to share the model on a specific organization page, you can join the organization and upload the model to the organization page (to do so, you need to specify the organization name in the `push_to_hub` function, e.g. `finetuned_model.push_to_hub(f"OrganizationName/{MODEL_NAME}", use_temp_dir=True)`).

### **Question 5: LLM-based summarization**

Thanks to their broad general knowledge and large-scale pretraining, LLMs can generate high-quality summaries in a zero-shot fashion (i.e., without any fine-tuning). Try using an LLM of your choice to produce the summaries.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from tqdm.auto import tqdm
import pandas as pd
import evaluate

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_name = "microsoft/Phi-4-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

def create_prompt(article_text):
    messages = [
        {"role": "user", "content": f"Summarize the following news article in about 3 sentences:\n\n{article_text}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

llm_summaries = []

subset_test = test_df.head(20)

for _, row in tqdm(subset_test.iterrows(), total=len(subset_test)):
    prompt = create_prompt(row['text'])
    
    outputs = pipe(prompt, return_full_text=False)
    generated_text = outputs[0]['generated_text'].strip()
    
    llm_summaries.append(generated_text)

rouge = evaluate.load('rouge')
results_llm = rouge.compute(
    predictions=llm_summaries,
    references=subset_test['summary'].tolist()
)

print(results_llm)

**Bonus**: Try to guide the final result:
- *Extractive summary*: Ask the LLM to generate a summary by selecting sentences from the original text. Is it capable of extracting them while keeping the original wording?
- *Style*: Use the "system" message or provide additional instruction in the initial prompt to shape the style of the genaration. Is the model able to produce both an informal and a formal summary?

In [ ]:
# your code here
def create_styled_prompt(article_text, style="standard"):
    
    # Definiamo istruzioni diverse in base allo stile richiesto
    if style == "extractive":
        instruction = (
            "Select and extract exactly 3 sentences from the text below that best summarize it. "
            "Do not paraphrase, use the original wording verbatim."
        )
    elif style == "informal":
        instruction = (
            "Summarize the following article for a text message to a friend. "
            "Use slang, be super casual, use emojis, and keep it very short."
        )
    elif style == "formal":
        instruction = (
            "Draft a professional executive summary of the following news report. "
            "Use formal business language, passive voice where appropriate, and maintain a neutral tone."
        )
    else: # standard
        instruction = "Summarize the following news article in about 3 sentences."

    messages = [
        {"role": "user", "content": f"{instruction}\n\nArticle:\n{article_text}"}
    ]
    
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

sample_article = test_df['text'].iloc[0]
styles = ["extractive", "informal", "formal"]

for s in styles:
    print(f"\nGenerating style: {s.upper()}...")
    prompt = create_styled_prompt(sample_article, style=s)

    outputs = pipe(prompt, return_full_text=False, max_new_tokens=150)
    print(f"OUTPUT:\n{outputs[0]['generated_text'].strip()}\n")
    print("-" * 30)

Evaluate the performance of the LLM-based summarization approach using the different prompts tested and compare it with the results obtained in the previous exercises (Q2, Q3, and Q4).

In [ ]:
# your code here